# ML Baseline Models: Image-Equivalent Numeric Inputs

This notebook trains and evaluates the non-image baselines on exactly the same samples and source data used by the CNN. It uses `image_manifest.pkl` as the sample source, `merged_df.pkl` as the numeric data source, and `feature_columns.json` to determine the exact 30-candle image-source feature window.


## 1. Setup

In [ ]:
from pathlib import Path
import sys
import random
import time
import copy
import json
from contextlib import nullcontext

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "cnnfin_5m.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing configs/cnnfin_5m.yaml")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cnnfin.config import load_config
from cnnfin.metrics import LABELS, bootstrap_macro_f1_ci, evaluate_predictions
from cnnfin.utils import ensure_dir, set_global_seed, write_json

CONFIG_PATH = ROOT / "configs" / "cnnfin_5m.yaml"
base_config = load_config(CONFIG_PATH)
artifact_dir = Path(base_config.artifact_dir)
if not artifact_dir.is_absolute():
    artifact_dir = ROOT / artifact_dir
config = load_config(CONFIG_PATH, artifact_dir=str(artifact_dir))

ARTIFACT_DIR = Path(config.artifact_dir)
PROCESSED_DIR = ARTIFACT_DIR / "processed"
RESULTS_DIR = ARTIFACT_DIR / "results"
MODEL_DIR = ARTIFACT_DIR / "models"
MERGED_PATH = PROCESSED_DIR / "merged_df.pkl"
FEATURE_COLUMNS_PATH = PROCESSED_DIR / "feature_columns.json"
IMAGE_MANIFEST_PATH = PROCESSED_DIR / "image_manifest.pkl"

# Set True for a quick smoke test only. Full Jarvis runs should keep this False.
DEBUG_MODE = False
DEBUG_ROWS_PER_CLASS = 32
DEBUG_NUM_EPOCHS = 1
DEBUG_BOOTSTRAP_ITERATIONS = 50

SEED = int(config.seeds[0])
set_global_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
PIN_MEMORY = DEVICE.type == "cuda"
NUM_WORKERS = int(config.num_workers)
NUM_EPOCHS = DEBUG_NUM_EPOCHS if DEBUG_MODE else int(config.num_epochs)
BOOTSTRAP_ITERATIONS = DEBUG_BOOTSTRAP_ITERATIONS if DEBUG_MODE else int(config.bootstrap_iterations)

ensure_dir(RESULTS_DIR)
ensure_dir(MODEL_DIR)

print(f"Repo root: {ROOT}")
print(f"Config: {CONFIG_PATH}")
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Merged df: {MERGED_PATH}")
print(f"Feature columns: {FEATURE_COLUMNS_PATH}")
print(f"Image manifest: {IMAGE_MANIFEST_PATH}")
print(f"Device: {DEVICE}")
print(f"Batch size: {config.batch_size}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Bootstrap iterations: {BOOTSTRAP_ITERATIONS}")
print(f"DEBUG_MODE: {DEBUG_MODE}")


## 2. Load CNN-Matched Samples And Source Features

In [ ]:
for required_path in [MERGED_PATH, FEATURE_COLUMNS_PATH, IMAGE_MANIFEST_PATH]:
    if not required_path.exists():
        if required_path == IMAGE_MANIFEST_PATH:
            raise FileNotFoundError(
                f"Missing {IMAGE_MANIFEST_PATH}. Run exploration/image_builder.ipynb with RUN_FULL_IMAGE_BUILD = True first."
            )
        raise FileNotFoundError(f"Missing required artifact: {required_path}")

merged_df = pd.read_pickle(MERGED_PATH).copy()
merged_df["Open time"] = pd.to_datetime(merged_df["Open time"], utc=True)
manifest = pd.read_pickle(IMAGE_MANIFEST_PATH).copy()
manifest["Open time"] = pd.to_datetime(manifest["Open time"], utc=True)
feature_columns = json.loads(FEATURE_COLUMNS_PATH.read_text())

source_cols = feature_columns["image_source_feature_cols"]
lookback = int(feature_columns["model_window_lookback"])
expected_dim = lookback * len(source_cols)

required_manifest_cols = {"sample_id", "Open time", "row_idx", "split", "label"}
missing_manifest_cols = sorted(required_manifest_cols - set(manifest.columns))
if missing_manifest_cols:
    raise ValueError(f"image_manifest.pkl is missing required columns: {missing_manifest_cols}")

missing_source_cols = [col for col in source_cols if col not in merged_df.columns]
if missing_source_cols:
    raise ValueError(f"merged_df.pkl is missing source feature columns: {missing_source_cols}")

manifest["label"] = manifest["label"].astype(int)
manifest["row_idx"] = manifest["row_idx"].astype(int)
manifest = manifest.sort_values(["Open time", "sample_id"]).reset_index(drop=True)

splits = {
    split: manifest[manifest["split"].eq(split)].copy().reset_index(drop=True)
    for split in ["train", "val", "test"]
}

if DEBUG_MODE:
    def debug_limit(frame: pd.DataFrame) -> pd.DataFrame:
        parts = []
        for label in LABELS:
            part = frame[frame["label"].eq(label)].head(DEBUG_ROWS_PER_CLASS)
            if len(part) == 0:
                raise ValueError(f"DEBUG_MODE requested label {label}, but split has no rows for it.")
            parts.append(part)
        return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    splits = {split: debug_limit(frame) for split, frame in splits.items()}
    print("DEBUG_MODE is True: using balanced limited samples per split.")

summary_rows = []
for split, frame in splits.items():
    if len(frame) == 0:
        raise ValueError(f"Split {split} is empty.")
    label_counts = frame["label"].value_counts().sort_index().to_dict()
    summary_rows.append({"split": split, "rows": len(frame), **{f"label_{k}": label_counts.get(k, 0) for k in LABELS}})

print(f"Image-equivalent numeric window: {lookback} candles x {len(source_cols)} features = {expected_dim} tabular features")
print("Source columns:")
print(source_cols)
display(pd.DataFrame(summary_rows))


## 3. Build Image-Equivalent Numeric Arrays

In [ ]:
def build_sequence_array(features: pd.DataFrame, samples: pd.DataFrame, columns: list[str], window: int) -> np.ndarray:
    values = features[columns].to_numpy(dtype=np.float32, copy=False)
    row_idx = samples["row_idx"].astype(int).to_numpy()
    out = np.empty((len(samples), window, len(columns)), dtype=np.float32)
    for i, end in enumerate(row_idx):
        start = int(end) - window + 1
        if start < 0:
            raise ValueError(f"Sample {samples.iloc[i]['sample_id']} has row_idx={end}, shorter than lookback={window}")
        out[i] = values[start : int(end) + 1]
    return out

start_time = time.time()
X_seq_raw = {split: build_sequence_array(merged_df, frame, source_cols, lookback) for split, frame in splits.items()}
X_tab_raw = {split: X_seq_raw[split].reshape(len(splits[split]), -1) for split in splits}
y = {split: splits[split]["label"].astype(int).to_numpy() for split in splits}

for split in ["train", "val", "test"]:
    print(f"{split}: X_seq={X_seq_raw[split].shape}, X_tab={X_tab_raw[split].shape}, y={y[split].shape}")
print(f"Built numeric arrays in {time.time() - start_time:.1f}s")


## 4. Shared Evaluation Helpers

In [ ]:
def class_weights_np(y_train: np.ndarray) -> np.ndarray:
    counts = np.bincount(y_train.astype(int), minlength=len(LABELS)).astype(np.float32)
    total = counts.sum()
    weights = np.ones(len(LABELS), dtype=np.float32)
    for cls in LABELS:
        if counts[cls] > 0:
            weights[cls] = total / (len(LABELS) * counts[cls])
    return weights


def prediction_frame(samples: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray | None) -> pd.DataFrame:
    out = samples[["sample_id", "Open time", "split", "row_idx"]].copy().reset_index(drop=True)
    out["y_true"] = y_true.astype(int)
    out["y_pred"] = y_pred.astype(int)
    if y_proba is not None:
        for cls in LABELS:
            out[f"p_{cls}"] = y_proba[:, cls]
    return out


def save_outputs(model_name: str, split: str, predictions: pd.DataFrame, metrics: dict) -> None:
    out_dir = ensure_dir(RESULTS_DIR / model_name)
    predictions.to_pickle(out_dir / f"{split}_predictions.pkl")
    write_json(metrics, out_dir / f"{split}_metrics.json")
    if split == "test":
        cm = pd.DataFrame(
            metrics["confusion_matrix"],
            index=[config.class_names[i] for i in LABELS],
            columns=[config.class_names[i] for i in LABELS],
        )
        cm.to_pickle(out_dir / "test_confusion_matrix.pkl")


def evaluate_and_save_model(model_name: str, split: str, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray | None) -> dict:
    metrics = evaluate_predictions(y_true, y_pred, class_names=config.class_names, y_proba=y_proba)
    metrics["macro_f1_ci"] = bootstrap_macro_f1_ci(
        y_true,
        y_pred,
        iterations=BOOTSTRAP_ITERATIONS if split == "test" else min(200, BOOTSTRAP_ITERATIONS),
        seed=SEED,
    )
    preds = prediction_frame(splits[split], y_true, y_pred, y_proba)
    save_outputs(model_name, split, preds, metrics)
    return metrics


def display_test_confusion(model_name: str, metrics: dict) -> None:
    cm = np.asarray(metrics["confusion_matrix"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[config.class_names[i] for i in LABELS])
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    plt.title(f"{model_name} Test Confusion Matrix")
    plt.show()

class_weights = class_weights_np(y["train"])
class_weight_dict = {int(i): float(w) for i, w in enumerate(class_weights)}
print("Class weights:", class_weight_dict)


## 5. Logistic Regression

In [ ]:
model_name = "logistic_regression"
print(f"Training {model_name}")
start = time.time()
logistic_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]
)
logistic_model.fit(X_tab_raw["train"], y["train"])
joblib.dump(logistic_model, MODEL_DIR / "logistic_regression.joblib")

val_pred = logistic_model.predict(X_tab_raw["val"]).astype(int)
val_proba = logistic_model.predict_proba(X_tab_raw["val"])
test_pred = logistic_model.predict(X_tab_raw["test"]).astype(int)
test_proba = logistic_model.predict_proba(X_tab_raw["test"])

val_metrics = evaluate_and_save_model(model_name, "val", y["val"], val_pred, val_proba)
test_metrics = evaluate_and_save_model(model_name, "test", y["test"], test_pred, test_proba)
print(f"{model_name} val_macro_f1={val_metrics['macro_f1']:.5f} test_macro_f1={test_metrics['macro_f1']:.5f} elapsed={time.time() - start:.1f}s")
display_test_confusion(model_name, test_metrics)


## 6. XGBoost

In [ ]:
model_name = "xgboost"
print(f"Training {model_name}")
start = time.time()
try:
    from xgboost import XGBClassifier

    xgb_imputer = SimpleImputer(strategy="median")
    Xgb_train = xgb_imputer.fit_transform(X_tab_raw["train"])
    Xgb_val = xgb_imputer.transform(X_tab_raw["val"])
    Xgb_test = xgb_imputer.transform(X_tab_raw["test"])
    sample_weight = np.asarray([class_weight_dict[int(label)] for label in y["train"]], dtype=np.float32)

    try:
        xgb_model = XGBClassifier(
            objective="multi:softprob",
            num_class=len(LABELS),
            eval_metric="mlogloss",
            n_estimators=1000,
            early_stopping_rounds=50,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
        )
        xgb_model.fit(
            Xgb_train,
            y["train"],
            sample_weight=sample_weight,
            eval_set=[(Xgb_val, y["val"])],
            verbose=False,
        )
    except TypeError:
        xgb_model = XGBClassifier(
            objective="multi:softprob",
            num_class=len(LABELS),
            eval_metric="mlogloss",
            n_estimators=300,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
        )
        xgb_model.fit(Xgb_train, y["train"], sample_weight=sample_weight, verbose=False)

    joblib.dump({"imputer": xgb_imputer, "model": xgb_model}, MODEL_DIR / "xgboost.joblib")

    val_pred = xgb_model.predict(Xgb_val).astype(int)
    val_proba = xgb_model.predict_proba(Xgb_val)
    test_pred = xgb_model.predict(Xgb_test).astype(int)
    test_proba = xgb_model.predict_proba(Xgb_test)

    val_metrics = evaluate_and_save_model(model_name, "val", y["val"], val_pred, val_proba)
    test_metrics = evaluate_and_save_model(model_name, "test", y["test"], test_pred, test_proba)
    print(f"{model_name} val_macro_f1={val_metrics['macro_f1']:.5f} test_macro_f1={test_metrics['macro_f1']:.5f} elapsed={time.time() - start:.1f}s")
    display_test_confusion(model_name, test_metrics)
except Exception as exc:
    write_json({"status": "skipped", "reason": str(exc)}, RESULTS_DIR / model_name / "test_metrics.json")
    raise


## 7. Shared PyTorch Training Helpers

In [ ]:
class ArrayDataset(Dataset):
    def __init__(self, x_array: np.ndarray, y_array: np.ndarray):
        self.x = torch.from_numpy(np.asarray(x_array, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y_array, dtype=np.int64))

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


def make_loader(x_array: np.ndarray, y_array: np.ndarray, *, shuffle: bool) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(SEED)
    kwargs = {
        "batch_size": int(config.batch_size),
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
        "generator": generator if shuffle else None,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
    return DataLoader(ArrayDataset(x_array, y_array), **kwargs)


def train_one_epoch_torch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    total_loss = 0.0
    total_rows = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
        yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        rows = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * rows
        total_rows += rows
    return total_loss / max(total_rows, 1)


@torch.no_grad()
def predict_torch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    model.eval()
    y_true_parts = []
    y_pred_parts = []
    y_proba_parts = []
    total_loss = 0.0
    total_rows = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
        yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        proba = torch.softmax(logits, dim=1)
        y_true_parts.append(yb.cpu().numpy())
        y_proba = proba.cpu().numpy()
        y_proba_parts.append(y_proba)
        y_pred_parts.append(y_proba.argmax(axis=1))
        rows = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * rows
        total_rows += rows
    return np.concatenate(y_true_parts), np.concatenate(y_pred_parts), np.concatenate(y_proba_parts), total_loss / max(total_rows, 1)


def fit_torch_classifier(model_name: str, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, test_loader: DataLoader):
    print(f"Training {model_name}")
    start = time.time()
    model = model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    best_state = None
    best_score = -1.0
    best_record = None
    bad_epochs = 0
    history_rows = []

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch_torch(model, train_loader, loss_fn, optimizer)
        val_true, val_pred, val_proba, val_loss = predict_torch(model, val_loader, loss_fn)
        val_metrics = evaluate_predictions(val_true, val_pred, class_names=config.class_names, y_proba=val_proba)
        record = {
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "val_accuracy": float(val_metrics["accuracy"]),
        }
        history_rows.append(record)
        print(f"{model_name} epoch={epoch} train_loss={train_loss:.5f} val_loss={val_loss:.5f} val_macro_f1={record['val_macro_f1']:.5f}")
        if record["val_macro_f1"] > best_score:
            best_score = record["val_macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            best_record = record.copy()
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= config.early_stopping_patience:
                print(f"Early stopping {model_name} after {bad_epochs} epochs without validation macro-F1 improvement.")
                break

    if best_state is None:
        raise RuntimeError(f"{model_name} did not produce a best checkpoint.")
    model.load_state_dict(best_state)

    model_dir = ensure_dir(MODEL_DIR)
    torch.save(
        {
            "model_state": model.state_dict(),
            "config": config.to_dict(),
            "best_val_macro_f1": float(best_score),
            "best_record": best_record,
            "source_cols": source_cols,
            "lookback": lookback,
        },
        model_dir / f"{model_name}.pt",
    )
    history = pd.DataFrame(history_rows)
    out_dir = ensure_dir(RESULTS_DIR / model_name)
    history.to_pickle(out_dir / "training_history.pkl")

    for split, frame, loader in [("val", splits["val"], val_loader), ("test", splits["test"], test_loader)]:
        true, pred, proba, loss = predict_torch(model, loader, loss_fn)
        metrics = evaluate_predictions(true, pred, class_names=config.class_names, y_proba=proba)
        metrics["loss"] = float(loss)
        metrics["macro_f1_ci"] = bootstrap_macro_f1_ci(
            true,
            pred,
            iterations=BOOTSTRAP_ITERATIONS if split == "test" else min(200, BOOTSTRAP_ITERATIONS),
            seed=SEED,
        )
        preds = prediction_frame(frame, true, pred, proba)
        save_outputs(model_name, split, preds, metrics)
        if split == "test":
            test_metrics = metrics

    print(f"{model_name} best_val_macro_f1={best_score:.5f} test_macro_f1={test_metrics['macro_f1']:.5f} elapsed={time.time() - start:.1f}s")
    display(history)
    display_test_confusion(model_name, test_metrics)
    return model, history, test_metrics


## 8. MLP

In [ ]:
# MLP uses the same flattened 30x26 image-source window as Logistic Regression and XGBoost.
mlp_imputer = SimpleImputer(strategy="median")
mlp_scaler = StandardScaler()
X_mlp_train = mlp_scaler.fit_transform(mlp_imputer.fit_transform(X_tab_raw["train"])).astype(np.float32)
X_mlp_val = mlp_scaler.transform(mlp_imputer.transform(X_tab_raw["val"])).astype(np.float32)
X_mlp_test = mlp_scaler.transform(mlp_imputer.transform(X_tab_raw["test"])).astype(np.float32)
joblib.dump({"imputer": mlp_imputer, "scaler": mlp_scaler, "source_cols": source_cols, "lookback": lookback}, MODEL_DIR / "mlp_preprocess.joblib")

class TabularMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, len(LABELS)),
        )

    def forward(self, x):
        return self.net(x)

mlp_train_loader = make_loader(X_mlp_train, y["train"], shuffle=True)
mlp_val_loader = make_loader(X_mlp_val, y["val"], shuffle=False)
mlp_test_loader = make_loader(X_mlp_test, y["test"], shuffle=False)

mlp_model, mlp_history, mlp_test_metrics = fit_torch_classifier(
    "mlp",
    TabularMLP(input_dim=expected_dim),
    mlp_train_loader,
    mlp_val_loader,
    mlp_test_loader,
)


## 9. LSTM

In [ ]:
# LSTM uses the same source features as a 30-step sequence instead of a flattened vector.
lstm_imputer = SimpleImputer(strategy="median")
lstm_scaler = StandardScaler()
train_rows = X_seq_raw["train"].reshape(-1, len(source_cols))
lstm_scaler.fit(lstm_imputer.fit_transform(train_rows))


def transform_sequence_split(x_seq: np.ndarray) -> np.ndarray:
    flat = x_seq.reshape(-1, len(source_cols))
    scaled = lstm_scaler.transform(lstm_imputer.transform(flat)).astype(np.float32)
    return scaled.reshape(x_seq.shape)

X_lstm_train = transform_sequence_split(X_seq_raw["train"])
X_lstm_val = transform_sequence_split(X_seq_raw["val"])
X_lstm_test = transform_sequence_split(X_seq_raw["test"])
joblib.dump({"imputer": lstm_imputer, "scaler": lstm_scaler, "source_cols": source_cols, "lookback": lookback}, MODEL_DIR / "numeric_lstm_preprocess.joblib")

class NumericLSTM(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_size=128, num_layers=1, batch_first=True)
        self.head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, len(LABELS)),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

lstm_train_loader = make_loader(X_lstm_train, y["train"], shuffle=True)
lstm_val_loader = make_loader(X_lstm_val, y["val"], shuffle=False)
lstm_test_loader = make_loader(X_lstm_test, y["test"], shuffle=False)

lstm_model, lstm_history, lstm_test_metrics = fit_torch_classifier(
    "numeric_lstm",
    NumericLSTM(input_dim=len(source_cols)),
    lstm_train_loader,
    lstm_val_loader,
    lstm_test_loader,
)


## 10. Summary

In [ ]:
summary_rows = []
for model_name in ["logistic_regression", "xgboost", "mlp", "numeric_lstm"]:
    metrics_path = RESULTS_DIR / model_name / "test_metrics.json"
    val_metrics_path = RESULTS_DIR / model_name / "val_metrics.json"
    if not metrics_path.exists():
        continue
    test_metrics = json.loads(metrics_path.read_text())
    val_metrics = json.loads(val_metrics_path.read_text()) if val_metrics_path.exists() else {}
    summary_rows.append(
        {
            "model": model_name,
            "val_macro_f1": val_metrics.get("macro_f1"),
            "test_macro_f1": test_metrics.get("macro_f1"),
            "test_accuracy": test_metrics.get("accuracy"),
            "test_weighted_f1": test_metrics.get("weighted_f1"),
            "macro_f1_ci_low": test_metrics.get("macro_f1_ci", {}).get("low"),
            "macro_f1_ci_high": test_metrics.get("macro_f1_ci", {}).get("high"),
        }
    )

model_summary = pd.DataFrame(summary_rows).sort_values("test_macro_f1", ascending=False, na_position="last")
model_summary.to_pickle(RESULTS_DIR / "ml_model_summary.pkl")
display(model_summary)
print(f"Saved ML model summary: {RESULTS_DIR / 'ml_model_summary.pkl'}")


## 11. Acceptance Checks

In [ ]:
required = []
for model_name in ["logistic_regression", "xgboost", "mlp", "numeric_lstm"]:
    required.extend(
        [
            RESULTS_DIR / model_name / "val_predictions.pkl",
            RESULTS_DIR / model_name / "test_predictions.pkl",
            RESULTS_DIR / model_name / "val_metrics.json",
            RESULTS_DIR / model_name / "test_metrics.json",
            RESULTS_DIR / model_name / "test_confusion_matrix.pkl",
        ]
    )
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected output artifacts: {missing}")

for model_name in ["logistic_regression", "xgboost", "mlp", "numeric_lstm"]:
    preds = pd.read_pickle(RESULTS_DIR / model_name / "test_predictions.pkl")
    if len(preds) != len(splits["test"]):
        raise ValueError(f"{model_name} test predictions rows {len(preds):,} != test manifest rows {len(splits['test']):,}")

print("ML baseline notebook completed successfully.")
print(f"All models used the same samples from: {IMAGE_MANIFEST_PATH}")
print(f"All models used the same source columns from: {FEATURE_COLUMNS_PATH}")
print(f"Numeric input surface: {lookback} x {len(source_cols)} = {expected_dim}")
